In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
import h5py
from pathlib import Path

In [ ]:
base_dir      = Path("/Users/angelaliu/Library/CloudStorage/OneDrive-MassachusettsInstituteofTechnology/Fracture Flow UROP")
transport_dir = base_dir / "Updated Code" / "transport"
mat_path = transport_dir / "compare_Np.mat"

In [ ]:
def load_compare_Np(mat_path):
    """
    Load compare_Np.mat (MATLAB v7.3 / HDF5).
    MATLAB (M, N) arrays/cells are stored transposed as (N, M) in HDF5.
    bkt_1000{ps, tr} (1-indexed MATLAB) -> f['bkt_1000'][tr-1, ps-1] (0-indexed HDF5).
    Returns:
      bkt       : dict with keys 'bkt_1000' and 'bkt_10000',
                  each a list-of-lists bkt[ps][tr] = 1-D float array (empty if disconnected)
      connected : (5, 5) bool array, connected[ps, tr]
      weights   : (5, 3) float array
    """
    with h5py.File(mat_path, 'r') as f:
        weights   = f['weights'][()].T           # MATLAB (5,3) -> HDF5 (3,5) -> .T = (5,3)
        connected = f['connected'][()].T.astype(bool)  # MATLAB (5,5) -> HDF5 (5,5) -> .T
        bkt = {}
        for key in ('bkt_1000', 'bkt_10000'):
            refs = f[key]                        # HDF5 (5, 5) refs; refs[tr, ps] = MATLAB {ps+1, tr+1}
            arr  = [[None] * 5 for _ in range(5)]
            for ps in range(5):
                for tr in range(5):
                    a = f[refs[tr, ps]][()]
                    if a.dtype == np.float64 and a.size > 0:
                        arr[ps][tr] = a.flatten()
                    else:
                        arr[ps][tr] = np.array([], dtype=np.float64)
            bkt[key] = arr
    return bkt, connected, weights


def recover_theta(w):
    disc = 1.0    + (5.0    - 1.0   ) * w[0]
    lr   = 0.3    + (0.8    - 0.3   ) * w[1]
    b    = 0.01e-3 + (0.3e-3 - 0.01e-3) * w[2]
    return disc, lr, b


bkt, connected, weights = load_compare_Np(mat_path)
print(f"connected matrix:\n{connected.astype(int)}")
print(f"weights shape: {weights.shape}")

In [ ]:
pdf_path = transport_dir / 'compare_Np.pdf'

with PdfPages(pdf_path) as pdf:
    for ps in range(5):
        disc, lr, b = recover_theta(weights[ps])
        for tr in range(5):
            is_conn = connected[ps, tr]

            fig, axes = plt.subplots(1, 2, figsize=(12, 5), sharex=True, sharey=True)
            fig.suptitle(
                f'PS {ps+1}, trial {tr+1}  |  '
                f'disc={disc:.2f}, lr={lr:.2f}, b={b:.2e}  |  '
                f'{"connected" if is_conn else "disconnected"}',
                fontsize=11)

            # compute shared x range from both particle counts
            all_bkts = [bkt[k][ps][tr] for k in ('bkt_1000', 'bkt_10000')
                        if len(bkt[k][ps][tr]) > 0]

            for ax, key, label in zip(
                    axes,
                    ('bkt_1000', 'bkt_10000'),
                    ('Np = 1 000', 'Np = 10 000')):

                bkt_arr = bkt[key][ps][tr]
                if len(bkt_arr) == 0:
                    ax.text(0.5, 0.5, 'disconnected',
                            ha='center', va='center',
                            transform=ax.transAxes, fontsize=13)
                    ax.set_xscale('log'); ax.set_yscale('log')
                else:
                    n    = len(bkt_arr)
                    surv = (n - np.arange(1, n + 1)) / n
                    ax.loglog(bkt_arr, surv, 'k.', ms=1.5, alpha=0.4)

                ax.set_title(label, fontsize=11)
                ax.set_xlabel('BKT (s)')
                ax.grid(True, which='both', alpha=0.3)

            axes[0].set_ylabel('1 - CDF')

            if all_bkts:
                x_min = min(a[0]  for a in all_bkts) * 0.9
                x_max = max(a[-1] for a in all_bkts) * 1.1
                axes[0].set_xlim(x_min, x_max)  # propagates to axes[1] via sharex

            plt.tight_layout()
            pdf.savefig(fig, bbox_inches='tight')
            plt.close(fig)

print(f'Saved {pdf_path}  (25 pages)')